In [50]:
from astroquery.heasarc import Heasarc
from astropy.coordinates import SkyCoord
import astropy.units as u
from datetime import datetime
from astropy.time import Time
import time
import pandas as pd

# List of transients with RA/Dec in decimal degrees
transients = [
	{"name": "AT2020ukj", "ra": 112.8925, "dec": 57.274733, "date": "2020-12-12"},
	{"name": "AT2023adr", "ra": 219.082625, "dec": 32.387905, "date": "2023-01-30"},
	{"name": "AT2023npi", "ra": 3.0655417, "dec": 22.321761, "date": "2023-01-01"}, # Made-up date
	{"name": "AT2020aexc", "ra": 181.805208, "dec": 44.170999, "date":"2021-02-22" },
	{"name": "ZTF20aatpzog", "ra": 169.726417, "dec": 21.828455, "date": "2020-04-08"},
	{"name": "AT2020afap", "ra": 174.140917, "dec": 61.7275, "date": "2021-03-01"},
	{"name": "AT2020pno", "ra": 251.654333, "dec": 55.607447, "date": "2020-08-01"},
	{"name": "AT2023jag", "ra": 242.300375, "dec": 46.561778, "date": "2023-06-12"},
	{"name": "AT2021ovg", "ra": 267.020875, "dec": 40.238172, "date": "2021-06-19"},
	{"name": "ZTF23abjvojy", "ra": 38.662667, "dec": 1.127817, "date": "2008-01-22"},
	{"name": "AT2022yhf", "ra": 326.528542, "dec": -11.96695, "date": "2023-01-02"},
	{"name": "AT2024nxp", "ra": 262.863458, "dec": 57.616136, "date": "2024-06-23"},
	{"name": "AT2020actc", "ra": 150.818, "dec": 21.095014, "date": "2021-01-25"},
	{"name": "AT2021wxd", "ra": 239.580, "dec": 7.473497, "date": "2021-09-16"},
	{"name": "AT2024hhj", "ra": 132.154667, "dec": 33.837503, "date": "2024-05-07"},
	{"name": "AT2023rav", "ra": 357.811208, "dec": 25.697403, "date": "2023-09-02"},
	{"name": "AT2023zaj", "ra": 9.3935, "dec": -13.564306, "date": "2023-01-01"}, # Made-up date
	{"name": "AT2023szj", "ra": 221.616667, "dec": 8.693528, "date": "2024-01-02"},
	{"name": "AT2024gzn", "ra": 268.817125, "dec": 38.978178, "date": "2024-06-02"},
	{"name": "AT2021lnu", "ra": 233.077833, "dec": -0.000206, "date": "2021-06-25"},
	{"name": "AT2020mvg", "ra": 181.360542, "dec": 28.909544, "date": "2020-06-12"},
	{"name": "AT2019agc", "ra": 211.688833, "dec": 13.918028, "date": "2019-01-01"}, # Made-up date
	{"name": "AT2024ljd", "ra": 318.4125, "dec": 2.493497, "date": "2024-06-26"},
]


heasarc = Heasarc()
radius = 5 * u.arcmin  # Search radius
results_summary = []

for obj in transients:
	print(f"Checking {obj['name']}...")
	skypos = SkyCoord(ra=obj['ra']*u.deg, dec=obj['dec']*u.deg)
	try:
		res = heasarc.query_region(skypos, radius=radius, catalog="swiftmastr")
	except Exception as e:
		print(f"  Error querying {obj['name']}: {e}")
		time.sleep(1)
		continue


	if res is not None and len(res) > 0:
		has_uvot = res['uvot_exposure'] > 0
		matching = res[has_uvot]
		# Filter by date
		matching = matching[Time(matching['start_time'], format='mjd') > Time(obj["date"], format="iso")]

		if len(matching) > 0:
			print(f"  ✅ UVOT data found for {obj['name']} after given date (peak)")
			# Compute delay in days
			obs_times = Time(matching["start_time"], format="mjd")
			t_date = Time(obj["date"], format="iso")
			time_deltas = (obs_times - t_date).to_value("day")

			for row, delta, t_iso in zip(matching, time_deltas, obs_times.iso):
				results_summary.append({
					"name": obj["name"],
					"transient_ra": obj["ra"],
					"transient_dec": obj["dec"],
					"obs_ra": row["ra"],
					"obs_dec": row["dec"],
					"obsid": row["obsid"],
					"start_time_mjd": row["start_time"],
					"start_time_iso": t_iso,
					"exposure": row["uvot_exposure"],
					"days_since_peak": round(delta, 2),
				})
			
				# links = heasarc.locate_data(res)
				# heasarc.download_data(links, location = f"./swift_data/{row['obsid']}")



		else:
			print(f"  ❌ No UVOT observations in catalog entries for {obj['name']}")
	else:
		print(f"  ❌ No Swift observations found for {obj['name']}")

# Save results
df = pd.DataFrame(results_summary)
df.to_csv("uvot_matches.csv", index=False)
print("\n✅ Saved UVOT-matched transients to 'uvot_matches.csv'")


Checking AT2020ukj...
  ✅ UVOT data found for AT2020ukj after given date (peak)
Checking AT2023adr...
  ✅ UVOT data found for AT2023adr after given date (peak)
Checking AT2023npi...


  ❌ No Swift observations found for AT2023npi
Checking AT2020aexc...
  ✅ UVOT data found for AT2020aexc after given date (peak)
Checking ZTF20aatpzog...


  ❌ No Swift observations found for ZTF20aatpzog
Checking AT2020afap...


  ❌ No Swift observations found for AT2020afap
Checking AT2020pno...
  ✅ UVOT data found for AT2020pno after given date (peak)
Checking AT2023jag...


  ❌ No Swift observations found for AT2023jag
Checking AT2021ovg...


...

  ❌ No Swift observations found for AT2023szj
Checking AT2024gzn...


  ❌ No Swift observations found for AT2024gzn
Checking AT2021lnu...


  ❌ No Swift observations found for AT2021lnu
Checking AT2020mvg...


  ❌ No Swift observations found for AT2020mvg
Checking AT2019agc...


  ❌ No Swift observations found for AT2019agc
Checking AT2024ljd...
  ❌ No Swift observations found for AT2024ljd

✅ Saved UVOT-matched transients to 'uvot_matches.csv'


In [48]:
links = heasarc.locate_data(res)
heasarc.download_data(links)

INFO: Downloading data from the HEASARC ... [astroquery.heasarc.core]
INFO: Downloading to ./heasarc-data.tar ... [astroquery.heasarc.core]
INFO: Untar ./heasarc-data.tar to . ... [astroquery.heasarc.core]
